# ACLCrossAttentionNet -- Unified Thesis Notebook

**A Cross-Attention Based Deep Learning Architecture for Optimal Fusion of Multi-Planar MRI Data in ACL Tear Detection**


## Figure/table -> thesis section map

| Output file | Thesis reference |
|---|---|
| `fig_3-1_class_balance.png` | Figure 3-1 |
| `fig_3-2_crop_demo_all_planes.png` | Figure 3-2 |
| `table_3-2_parameter_breakdown.csv` | Table 3-2 |
| `fig_4-1_proposed_training_curve.png` | Figure 4-1 |
| `table_4-1_baseline_comparison.csv` | Table 4-1 / 4-4 |
| `fig_4-2_metrics_comparison.png` | Figure 4-2 |
| `fig_4-3_auc_confidence_intervals.png` | Figure 4-3 |
| `fig_4-4_gradcam_3plane_case.png` | Figure 4-4 |
| `fig_4-5_gradcam_multicase_grid.png` | Figure 4-5 |
| `fig_4-6_gradcam_failure_case.png` | Figure 4-6 |
| `fig_4-7_slice_attention_weights.png` | Figure 4-7 |
| `fig_4-8_params_comparison.png` | Figure 4-8 |
| `table_4-3_ablation_detail.csv` | Table 4-3 |
| `THRESHOLD_OPTIMIZATION printout` | fills the open bracket in section 4.3 |

Run cells top to bottom. Training all 8 model variants on a GTX 1650 (4GB) takes roughly the same order
of magnitude as before (~1500 minutes / ~25 hours total, per the previous run's own timing) -- this has
not changed, since the model sizes and epoch budget are the same.


## 0. Setup, Reproducibility, and Official Configuration

Constants below are the values **decided as official for the thesis** -- see thesis section 3.5/3.6 and the note at the top of this notebook .

In [ ]:
import os, random, warnings, math, time, gc, functools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, roc_curve, confusion_matrix, classification_report, auc as sk_auc
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42

def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # benchmark=True keeps cuDNN's fast auto-tuned kernels (needed on a 4GB card);
    # this trades a small amount of bit-for-bit determinism for speed (thesis 3.6).
    torch.backends.cudnn.benchmark = True

set_global_seed(SEED)

# ============================================================================
# OFFICIAL constants (thesis sections 3.2-3.6) -- do not silently change these;
# every number in the thesis text assumes this exact configuration.
# ============================================================================
BASE_DIR            = r"C:\jupyter\Dataset\MRNet-v1.0"
BATCH_SIZE           = 1
ACCUMULATION_STEPS   = 8              # effective batch = 8 (thesis 3.5.3)
MAX_SLICES           = 16
IMG_SIZE             = 224
CROP_SIZE            = 102            # keeps 39.8% of 256x256 (SFM, thesis 3.4.3)
EPOCHS               = 15             # official run length (thesis 3.5.2)
PATIENCE             = 3 

LR_HEAD              = 1e-4
LR_BACKBONE          = 1e-5
WD_BACKBONE          = 1e-5 
WD_HEAD              = 1e-4 

LAMBDA_CLINICAL      = 0.10           # gate KL-divergence weight (thesis 3.5.1)
LAMBDA_FOCAL         = 0.40           # focal-loss weight inside CombinedLoss
LAMBDA_ENT           = 0.005          # gate entropy-regularization weight
CLINICAL_GATE_TARGET = (0.50, 0.30, 0.20)   # Sagittal / Coronal / Axial prior (thesis 3.5.1)

VIEW_DROPOUT_P       = 0.1            # OFFICIAL value -- see thesis 3.4.4 note on the doc/code mismatch

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_TYPE = "cuda" if torch.cuda.is_available() else "cpu"

FIG_DIR = "thesis_outputs"
os.makedirs(FIG_DIR, exist_ok=True)

print(f"[Seed]   Global seed fixed at {SEED}")
print(f"[Config] Device            : {DEVICE}")
if torch.cuda.is_available():
    print(f"[Config] GPU               : {torch.cuda.get_device_name(0)}")
    print(f"[Config] VRAM              : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"[Config] CenterCrop        : {CROP_SIZE}x{CROP_SIZE} px ({CROP_SIZE/256*100:.1f}% of 256x256)")
print(f"[Config] Effective batch   : {BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"[Config] Weight decay      : backbone={WD_BACKBONE}, head={WD_HEAD}")
print(f"[Config] View Dropout      : {VIEW_DROPOUT_P}  ")
print(f"[Config] Epoch budget      : {EPOCHS} epochs, patience={PATIENCE}")
print(f"[Config] Outputs saved to  : ./{FIG_DIR}/")


## 1. Data Pipeline (thesis sections 3.2, 3.3)

Transforms and `MRNetDataset` are unchanged from both source notebooks. Thesis 3.3 describes this exact sequence: CenterCrop -> Resize -> [augmentation, train only] -> Normalize -> [Gaussian noise, train only].

In [ ]:
class AddGaussianNoise:
    def __init__(self, std=0.02): self.std = std
    def __call__(self, t):        return t + torch.randn_like(t) * self.std

# CROP_SIZE=102 -> 39.8% of 256x256 -> cuts ~60% surrounding (SFM, thesis 3.4.3)
train_transform = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=6, translate=(0.03, 0.03)),
    transforms.Normalize(mean=[0.485], std=[0.229]),
    AddGaussianNoise(std=0.02),
])

valid_transform = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Normalize(mean=[0.485], std=[0.229]),
])


class MRNetDataset(Dataset):
    """MRNet ACL dataset -- returns MAX_SLICES central slices from sagittal/coronal/axial."""
    def __init__(self, root_dir, task='acl', split='train', transform=None):
        self.root_dir  = root_dir
        self.split     = split
        self.transform = transform
        label_path     = os.path.join(root_dir, f"{split}-{task}.csv")
        self.labels    = pd.read_csv(label_path, names=['id', 'label'])

    def __len__(self):
        return len(self.labels)

    def _load_volume(self, case_id, view, source_split=None):
        sp = source_split if source_split is not None else self.split
        path = os.path.join(self.root_dir, sp, view, f"{str(case_id).zfill(4)}.npy")
        return np.load(path).astype(np.float32)

    def __getitem__(self, idx):
        folder_id = self.labels.iloc[idx, 0]
        label     = float(self.labels.iloc[idx, 1])
        view_tensors = []
        for view in ['sagittal', 'coronal', 'axial']:
            vol = self._load_volume(folder_id, view)
            mid   = vol.shape[0] // 2
            start = max(0, mid - MAX_SLICES // 2)
            vol   = vol[start : start + MAX_SLICES]
            if vol.shape[0] < MAX_SLICES:
                pad = MAX_SLICES - vol.shape[0]
                vol = np.pad(vol, ((0, pad), (0, 0), (0, 0)), mode='constant')
            vol = torch.from_numpy(vol) / 255.0
            slices = []
            for sl in vol:
                sl = sl.unsqueeze(0)
                if self.transform:
                    sl = self.transform(sl)
                slices.append(sl)
            view_tensors.append(torch.stack(slices).squeeze(1))  # (16, 224, 224)
        sag, cor, ax = view_tensors
        return sag, cor, ax, torch.tensor([label], dtype=torch.float32)

    def get_labels(self):
        return self.labels['label'].astype(float).tolist()


train_ds = MRNetDataset(BASE_DIR, task='acl', split='train', transform=train_transform)
valid_ds = MRNetDataset(BASE_DIR, task='acl', split='valid', transform=valid_transform)

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, generator=g)   # num_workers=0: Windows (thesis 3.6)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

pos_tr = int(train_ds.labels['label'].sum()); neg_tr = len(train_ds) - pos_tr
pos_vl = int(valid_ds.labels['label'].sum()); neg_vl = len(valid_ds) - pos_vl
print(f"[Dataset] Train : {len(train_ds)} samples | pos={pos_tr} ({pos_tr/len(train_ds)*100:.1f}%)  neg={neg_tr} ({neg_tr/len(train_ds)*100:.1f}%)")
print(f"[Dataset] Valid : {len(valid_ds)} samples | pos={pos_vl} ({pos_vl/len(valid_ds)*100:.1f}%)  neg={neg_vl} ({neg_vl/len(valid_ds)*100:.1f}%)")
print("[Dataset] These numbers should match thesis section 3.2 exactly (1130/120, 18.4%/45.0%).")


## 2. Dataset & Image Statistics -- EDA (produces Figures 3-1 and 3-2)

In [ ]:
# ---- Class balance chart -> Figure 3-1 -------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax_i, (name, pos, neg) in zip(axs, [('Train', pos_tr, neg_tr), ('Valid', pos_vl, neg_vl)]):
    bars = ax_i.bar(['Normal', 'Torn ACL'], [neg, pos], color=['steelblue', 'crimson'])
    total = pos + neg
    for b, v in zip(bars, [neg, pos]):
        ax_i.text(b.get_x() + b.get_width()/2, v + total*0.01, f"{v}\n({v/total*100:.1f}%)",
                   ha='center', fontsize=10)
    ax_i.set_title(f"{name} Set  (n={total})")
    ax_i.set_ylim(0, total * 0.85)
plt.suptitle("Class Balance -- Train vs Valid (MRNet-v1.0)", y=1.03, fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_3-1_class_balance.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_3-1_class_balance.png -> Thesis Figure 3-1")


In [ ]:
# ---- CenterCrop demonstration, ALL THREE PLANES IN ONE FIGURE ---------------
# section 3.3 (Figure 3-2) describes: raw + crop->resize, for all three planes.
norm_idx = valid_ds.labels[valid_ds.labels['label'] == 0].index[0]
torn_idx = valid_ds.labels[valid_ds.labels['label'] == 1].index[0]
demo_cid = valid_ds.labels.iloc[torn_idx, 0]

fig, axs = plt.subplots(2, 3, figsize=(13, 8))
for col, view in enumerate(['sagittal', 'coronal', 'axial']):
    raw_vol = valid_ds._load_volume(demo_cid, view)
    mid = raw_vol.shape[0] // 2
    raw_slice = raw_vol[mid]
    raw_h, raw_w = raw_slice.shape

    raw_t = torch.from_numpy(raw_slice).float().unsqueeze(0)
    resized_t = transforms.Resize((IMG_SIZE, IMG_SIZE))(transforms.CenterCrop(CROP_SIZE)(raw_t))

    axs[0, col].imshow(raw_slice, cmap='gray')
    half = CROP_SIZE / 2
    cy, cx = raw_h / 2, raw_w / 2
    rect = plt.Rectangle((cx - half, cy - half), CROP_SIZE, CROP_SIZE,
                          edgecolor='red', facecolor='none', lw=2)
    axs[0, col].add_patch(rect)
    axs[0, col].set_title(f"{view.capitalize()} -- Original ({raw_h}x{raw_w})")
    axs[0, col].axis('off')

    axs[1, col].imshow(resized_t.squeeze().numpy(), cmap='gray')
    axs[1, col].set_title(f"After Crop({CROP_SIZE})->Resize({IMG_SIZE})")
    axs[1, col].axis('off')

plt.suptitle(f"Preprocessing: CenterCrop + Resize on All Three Planes (kept {CROP_SIZE/256*100:.1f}%, "
             f"case #{demo_cid}, Torn ACL)", y=1.0, fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_3-2_crop_demo_all_planes.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_3-2_crop_demo_all_planes.png -> Thesis Figure 3-2 (no more overwrite bug)")


## 3. Architecture: ACLCrossAttentionNet (thesis sections 3.4.1-3.4.6)


In [ ]:
# --- 1. Multi-Scale Shared EfficientNet-B0 Backbone (thesis 3.4.1) -----------
class SharedEfficientNet(nn.Module):
    """
    Multi-scale feature extraction, SHARED across all three planes:
      shallow (features[0:5]) -> 14x14, 80 ch
      deep    (features[5:9]) ->  7x7, 1280 ch
    Projected & fused to 256 ch at 7x7 resolution.
    BatchNorm layers are kept in eval() mode during training (see train_epoch) --
    necessary because batch_size=1 makes BN statistics from a single sample
    unstable (thesis 3.4.1).
    """
    def __init__(self):
        super().__init__()
        b0    = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        feats = list(b0.features.children())
        self.shallow = nn.Sequential(*feats[:5])
        self.deep    = nn.Sequential(*feats[5:])
        self.proj_s  = nn.Conv2d(80,   128, 1)
        self.proj_d  = nn.Conv2d(1280, 128, 1)
        self.pool_s  = nn.AdaptiveAvgPool2d((7, 7))
        self.fuse    = nn.Conv2d(256,  256, 1)

    def forward(self, x):
        b, s, h, w = x.shape
        x  = x.view(b * s, 1, h, w).repeat(1, 3, 1, 1).float()   # 1ch -> 3ch (thesis 3.4.1)
        fs = self.shallow(x)
        fd = self.deep(fs)
        fs = self.pool_s(self.proj_s(fs))
        fd = self.proj_d(fd)
        return self.fuse(torch.cat([fs, fd], dim=1))


# --- 2. Spatial Tokenizer -----------------------------------------------------
class SpatialTokenizer(nn.Module):
    def __init__(self, num_tokens=49, embed_dim=256):
        super().__init__()
        self.pos_emb = nn.Parameter(torch.randn(1, 1, num_tokens, embed_dim) * 0.02)

    def forward(self, x, b, s):
        _, c, h, w = x.shape
        x = x.view(b, s, c, h * w).permute(0, 1, 3, 2)
        return (x + self.pos_emb).float()


# --- 3. Slice-Level Attention Pooling (thesis 3.4.5) --------------------------
class SpatiallyAwareSliceAttention(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(embed_dim, 64), nn.Tanh(), nn.Linear(64, 1))

    def forward(self, x):
        x = x.float()
        w = F.softmax(self.mlp(x), dim=1)
        return (x * w).sum(dim=1)


# --- 4. Cross-Attention Block [float32-safe] (thesis 3.4.2) -------------------
class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()
        self.mha   = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=0.1)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2), nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim * 2, embed_dim),
        )

    def forward(self, q, k, v):
        q, k, v = q.float(), k.float(), v.float()   # NaN guard under AMP
        a, _    = self.mha(q, k, v)
        x       = self.norm1(q + a)
        return  self.norm2(x + self.ffn(x))


class MultiPlanarCrossAttention(nn.Module):
    """Bidirectional cross-attention: each plane attends to both others (6 directions total)."""
    def __init__(self, embed_dim=256):
        super().__init__()
        self.sc  = CrossAttentionBlock(embed_dim)
        self.sa  = CrossAttentionBlock(embed_dim)
        self.cs  = CrossAttentionBlock(embed_dim)
        self.ca  = CrossAttentionBlock(embed_dim)
        self.as_ = CrossAttentionBlock(embed_dim)
        self.ac  = CrossAttentionBlock(embed_dim)

    def forward(self, s, c, a):
        s2 = s + self.sc(s, c, c) + self.sa(s, a, a)
        c2 = c + self.cs(c, s, s) + self.ca(c, a, a)
        a2 = a + self.as_(a, s, s) + self.ac(a, c, c)
        return s2, c2, a2


# --- 5. Adaptive Gated Fusion [gate-collapse resistant] (thesis 3.4.4) --------
class AdaptiveGatedFusion(nn.Module):
    """
    Defences against gate collapse to a single plane:
      - Zero-init      -> training starts from a uniform 1/3-1/3-1/3 prior
      - ViewDrop(p)    -> prevents over-reliance on any single plane
      - EntropyReg     -> applied in train_epoch via gate_entropy_reg()
    """
    def __init__(self, embed_dim=256, view_drop_p=VIEW_DROPOUT_P):
        super().__init__()
        self.gate      = nn.Linear(embed_dim * 3, 3)
        nn.init.zeros_(self.gate.weight)
        nn.init.zeros_(self.gate.bias)
        self.view_drop = nn.Dropout(p=view_drop_p)

    def forward(self, s, c, a):
        s  = self.view_drop(s.float())
        c  = self.view_drop(c.float())
        a  = self.view_drop(a.float())
        ctx = torch.cat([s.mean(1), c.mean(1), a.mean(1)], dim=-1)
        w   = F.softmax(self.gate(ctx), dim=-1)
        ws, wc, wa = w[:, 0:1, None], w[:, 1:2, None], w[:, 2:3, None]
        return s * ws + c * wc + a * wa, w


# --- 6. Context Encoder [float32 cast] (thesis 3.4.6) -------------------------
class SafeTransformerBlock(nn.Module):
    def __init__(self, embed_dim=256, nhead=4, ff_dim=512, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(embed_dim, nhead, batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, ff_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(ff_dim, embed_dim),
        )

    def forward(self, x):
        x = x.float()
        a, _ = self.attn(x, x, x)   # self-attention, not cross-attention (thesis 3.4.6)
        x = self.norm1(x + a)
        return self.norm2(x + self.ffn(x))


# --- 7. Classification Head (thesis 3.4.6) -------------------------------------
class MedicalClassificationHead(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128), nn.GELU(), nn.Dropout(0.35),
            nn.Linear(128, 64),        nn.GELU(), nn.Dropout(0.30),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.head(x.float())


# --- 8. Proposed Network -------------------------------------------------------
class ACLCrossAttentionNet(nn.Module):
    """A Cross-Attention Based Deep Learning Architecture for Optimal Fusion of
    Multi-Planar MRI Data in ACL Tear Detection."""
    def __init__(self):
        super().__init__()
        self.backbone      = SharedEfficientNet()
        self.tokenizer     = SpatialTokenizer()
        self.slice_attn    = SpatiallyAwareSliceAttention()
        self.cross_block_1 = MultiPlanarCrossAttention()
        self.cross_block_2 = MultiPlanarCrossAttention()
        self.gated_fusion   = AdaptiveGatedFusion()
        self.context_enc   = SafeTransformerBlock()
        self.classifier    = MedicalClassificationHead()

    def forward(self, sag, cor, ax):
        b, s, h, w = sag.shape
        fs = self.backbone(sag); fc = self.backbone(cor); fa = self.backbone(ax)
        ts = self.tokenizer(fs, b, s); tc = self.tokenizer(fc, b, s); ta = self.tokenizer(fa, b, s)
        ts = self.slice_attn(ts); tc = self.slice_attn(tc); ta = self.slice_attn(ta)
        ts, tc, ta = self.cross_block_1(ts, tc, ta)
        ts, tc, ta = self.cross_block_2(ts, tc, ta)
        fused, gate_w = self.gated_fusion(ts, tc, ta)
        ctx = self.context_enc(fused)
        vec = ctx.mean(dim=1)
        return self.classifier(vec), gate_w


# --- 9. Ablation: No-Cross-Attention Variant (thesis 4.7) ----------------------
class ACLNoCrossAttentionNet(nn.Module):
    """Identical to ACLCrossAttentionNet EXCEPT cross_block_1/cross_block_2 are
    removed. This IS the ablation study for thesis section 4.7 -- there is no
    separate ablation experiment elsewhere in this notebook."""
    def __init__(self):
        super().__init__()
        self.backbone     = SharedEfficientNet()
        self.tokenizer    = SpatialTokenizer()
        self.slice_attn   = SpatiallyAwareSliceAttention()
        self.gated_fusion = AdaptiveGatedFusion()
        self.context_enc  = SafeTransformerBlock()
        self.classifier   = MedicalClassificationHead()

    def forward(self, sag, cor, ax):
        b, s, h, w = sag.shape
        fs = self.backbone(sag); fc = self.backbone(cor); fa = self.backbone(ax)
        ts = self.tokenizer(fs, b, s); tc = self.tokenizer(fc, b, s); ta = self.tokenizer(fa, b, s)
        ts = self.slice_attn(ts); tc = self.slice_attn(tc); ta = self.slice_attn(ta)
        fused, gate_w = self.gated_fusion(ts, tc, ta)
        ctx = self.context_enc(fused)
        vec = ctx.mean(dim=1)
        return self.classifier(vec), gate_w


p_full = sum(p.numel() for p in ACLCrossAttentionNet().parameters())
p_abl  = sum(p.numel() for p in ACLNoCrossAttentionNet().parameters())
print("[Architecture] ACLCrossAttentionNet + ACLNoCrossAttentionNet defined.")
print(f"[Architecture] Full model params    : {p_full:,}")
print(f"[Architecture] Ablated model params : {p_abl:,}  "
      f"(cross-attention adds {p_full-p_abl:,} params, {(p_full-p_abl)/p_full*100:.1f}% of total)")


In [ ]:
# ---- Full per-module parameter breakdown -> Table 3-2 -----------------------
def count_parameters(model):
    rows = []
    for name, module in model.named_children():
        n_params = sum(p.numel() for p in module.parameters())
        rows.append({'Module': name, 'Parameters': n_params})
    df = pd.DataFrame(rows)
    df['% of Total'] = (df['Parameters'] / df['Parameters'].sum() * 100).round(1)
    return df

probe_model = ACLCrossAttentionNet().to(DEVICE)
param_table = count_parameters(probe_model)
print(param_table.to_string(index=False))
param_table.to_csv(f"{FIG_DIR}/table_3-2_parameter_breakdown.csv", index=False)
print(f"\n[Saved] table_3-2_parameter_breakdown.csv -> Thesis Table 3-2")
del probe_model
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 4. Loss Functions (thesis section 3.5.1)

In [ ]:
# --- Focal Loss ----------------------------------------------------------------
class FocalLoss(nn.Module):
    """Focuses training on hard examples; down-weights easy ones (thesis 3.5.1)."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        logits, targets = logits.float(), targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt  = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


# --- Combined Loss: (1-lambda)*BCE + lambda*Focal -------------------------------
class CombinedLoss(nn.Module):
    def __init__(self, pos_weight, focal_w=LAMBDA_FOCAL):
        super().__init__()
        self.bce_fn   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.focal_fn = FocalLoss(alpha=0.25, gamma=2.0)
        self.fw       = focal_w

    def forward(self, logits, targets):
        l, t = logits.float(), targets.float()
        return (1 - self.fw) * self.bce_fn(l, t) + self.fw * self.focal_fn(l, t)


# --- Clinical Gate Loss: KL(gate || target) -------------------------------------
class ClinicalGateLoss(nn.Module):
    """KL-divergence toward the clinically-motivated gate prior
    Sagittal=0.50, Coronal=0.30, Axial=0.20 (thesis 3.5.1)."""
    def __init__(self, target=CLINICAL_GATE_TARGET):
        super().__init__()
        self.register_buffer('target', torch.tensor(target, dtype=torch.float32))

    def forward(self, gate_w):
        t = self.target.expand_as(gate_w)
        return F.kl_div(gate_w.clamp(min=1e-8).log(), t, reduction='batchmean')


# --- Gate Entropy Regularizer ----------------------------------------------------
def gate_entropy_reg(gate_w, lam=LAMBDA_ENT):
    """Maximises entropy to discourage collapse to any single plane (thesis 3.5.1)."""
    entropy = -(gate_w * (gate_w + 1e-8).log()).sum(-1).mean()
    return -lam * entropy


clinical_loss_fn = ClinicalGateLoss().to(DEVICE)
print("[Loss] FocalLoss + CombinedLoss + ClinicalGateLoss + gate_entropy_reg defined.")


## 5. Baseline Model Definitions (thesis section 3.7)


In [ ]:
# --- Baseline: Single-plane nets -----------------------------------------------
class SinglePlaneNet(nn.Module):
    """Uses ONLY one MRI plane; fusion is simply absent (thesis 3.7)."""
    def __init__(self, plane='sagittal'):
        super().__init__()
        assert plane in ('sagittal', 'coronal', 'axial')
        self.plane       = plane
        self.backbone    = SharedEfficientNet()
        self.tokenizer   = SpatialTokenizer()
        self.slice_attn  = SpatiallyAwareSliceAttention()
        self.context_enc = SafeTransformerBlock()
        self.classifier  = MedicalClassificationHead()

    def forward(self, sag, cor, ax):
        x = {'sagittal': sag, 'coronal': cor, 'axial': ax}[self.plane]
        b, s, h, w = x.shape
        f   = self.backbone(x)
        t   = self.slice_attn(self.tokenizer(f, b, s))
        ctx = self.context_enc(t)
        vec = ctx.mean(dim=1)
        return self.classifier(vec), None


# --- Baseline: Naive concatenation fusion ---------------------------------------
class ConcatFusionNet(nn.Module):
    """Naive fusion via concatenation + linear projection instead of cross-attention."""
    def __init__(self):
        super().__init__()
        self.backbone    = SharedEfficientNet()
        self.tokenizer   = SpatialTokenizer()
        self.slice_attn  = SpatiallyAwareSliceAttention()
        self.fuse_proj   = nn.Linear(256 * 3, 256)
        self.context_enc = SafeTransformerBlock()
        self.classifier  = MedicalClassificationHead()

    def forward(self, sag, cor, ax):
        b, s, h, w = sag.shape
        fs, fc, fa = self.backbone(sag), self.backbone(cor), self.backbone(ax)
        ts = self.slice_attn(self.tokenizer(fs, b, s))
        tc = self.slice_attn(self.tokenizer(fc, b, s))
        ta = self.slice_attn(self.tokenizer(fa, b, s))
        fused = self.fuse_proj(torch.cat([ts, tc, ta], dim=-1))
        ctx = self.context_enc(fused)
        vec = ctx.mean(dim=1)
        return self.classifier(vec), None


# --- Baseline: Simple average fusion --------------------------------------------
class AverageFusionNet(nn.Module):
    """Unweighted mean of the three planes' token sequences."""
    def __init__(self):
        super().__init__()
        self.backbone    = SharedEfficientNet()
        self.tokenizer   = SpatialTokenizer()
        self.slice_attn  = SpatiallyAwareSliceAttention()
        self.context_enc = SafeTransformerBlock()
        self.classifier  = MedicalClassificationHead()

    def forward(self, sag, cor, ax):
        b, s, h, w = sag.shape
        fs, fc, fa = self.backbone(sag), self.backbone(cor), self.backbone(ax)
        ts = self.slice_attn(self.tokenizer(fs, b, s))
        tc = self.slice_attn(self.tokenizer(fc, b, s))
        ta = self.slice_attn(self.tokenizer(fa, b, s))
        fused = (ts + tc + ta) / 3.0
        ctx = self.context_enc(fused)
        vec = ctx.mean(dim=1)
        return self.classifier(vec), None


# --- Baseline: MRNet-style external reproduction (Bien et al., 2018) -----------
class MRNetStyleBaseline(nn.Module):
    """Approximate reproduction of the original MRNet architecture: per-view
    AlexNet features, max-pooled across slices, per-view logistic regression,
    combined by a final logistic regression -- the decision-level fusion
    described in thesis section 1.2. Reproduced at this notebook's fixed
    MAX_SLICES=16 rather than the original's variable per-case depth; trained
    through the SAME shared pipeline as every other row for a fair comparison
    (thesis 3.7)."""
    def __init__(self):
        super().__init__()
        def split_alexnet():
            alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
            feats = list(alexnet.features.children())
            return nn.Sequential(*feats[:8]), nn.Sequential(*feats[8:])
        shallow_list, deep_list = [], []
        for _ in range(3):
            sh, dp = split_alexnet()
            shallow_list.append(sh); deep_list.append(dp)
        self.backbone         = nn.Module()
        self.backbone.shallow = nn.ModuleList(shallow_list)
        self.backbone.deep    = nn.ModuleList(deep_list)
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))
        self.view_fc  = nn.ModuleList([nn.Linear(256, 1) for _ in range(3)])
        self.combine  = nn.Linear(3, 1)

    def _view_logit(self, x, idx):
        b, s, h, w = x.shape
        x = x.view(b * s, 1, h, w).repeat(1, 3, 1, 1).float()
        f = self.backbone.deep[idx](self.backbone.shallow[idx](x))
        f = self.pool(f).flatten(1)
        f = f.view(b, s, -1).max(dim=1).values
        return self.view_fc[idx](f)

    def forward(self, sag, cor, ax):
        ls = self._view_logit(sag, 0)
        lc = self._view_logit(cor, 1)
        la = self._view_logit(ax,  2)
        logit = self.combine(torch.cat([ls, lc, la], dim=-1))
        return logit, None


print("[Baselines] SinglePlaneNet / ConcatFusionNet / AverageFusionNet / MRNetStyleBaseline defined.")


## 6. Training Infrastructure (thesis sections 3.5.2, 3.5.3)

`autocast` is explicitly kept **disabled** below -- the forward pass runs in full float32, matching the thesis 3.5.3 note that numerical stability was prioritized over AMP speed given the float32-safety requirements inside the attention blocks (3.4.2, 3.4.6).

In [ ]:
def train_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    model.backbone.shallow.eval()   # BN stays in eval mode at batch_size=1 (thesis 3.4.1)
    model.backbone.deep.eval()

    total_loss = 0.0
    optimizer.zero_grad()

    all_probs, all_labels = [], []          # <-- NEW: for train AUC
    gate_sum, gate_n = None, 0              # <-- NEW: running weighted sum of gate weights

    for i, (s, c, a, lbl) in enumerate(loader):
        s, c, a, lbl = (s.to(DEVICE), c.to(DEVICE), a.to(DEVICE), lbl.to(DEVICE))
        lbl_smooth = lbl * 0.90 + 0.05   # label smoothing (thesis 3.5.1)

        with torch.amp.autocast(DEVICE_TYPE, enabled=False):   # autocast OFF -- thesis 3.5.3
            logits, gate_w = model(s, c, a)
            loss_cls  = criterion(logits, lbl_smooth)
            if gate_w is not None:
                loss_gate = LAMBDA_CLINICAL * clinical_loss_fn(gate_w)
                loss_ent  = gate_entropy_reg(gate_w)
            else:
                # Baselines with no meaningful 3-way gate (single-plane, naive
                # fusion, MRNet-style) skip the clinical-prior/entropy terms --
                # these are specific to the proposed gated design. Core
                # classification loss stays IDENTICAL for every model.
                loss_gate = torch.tensor(0.0, device=DEVICE)
                loss_ent  = torch.tensor(0.0, device=DEVICE)
            loss = (loss_cls + loss_gate + loss_ent) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS

        # -- NEW: bookkeeping for train AUC + gate weights (no grad, cheap) --
        with torch.no_grad():
            all_probs.extend(torch.sigmoid(logits.detach()).cpu().numpy().flatten())
            all_labels.extend(lbl.detach().cpu().numpy().flatten())

            if gate_w is not None:
                bsz = s.size(0)
                batch_gate = gate_w.detach().float().view(bsz, -1).mean(dim=0).cpu()  # (num_gates,)
                gate_sum = batch_gate * bsz if gate_sum is None else gate_sum + batch_gate * bsz
                gate_n += bsz

    train_auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else float('nan')
    avg_gate  = (gate_sum / gate_n).tolist() if gate_sum is not None else None

    return total_loss / len(loader), train_auc, avg_gate


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0
    for s, c, a, lbl in loader:
        s, c, a, lbl = (s.to(DEVICE), c.to(DEVICE), a.to(DEVICE), lbl.to(DEVICE))
        logits, _ = model(s, c, a)
        total_loss += criterion(logits.float(), lbl.float()).item()
        all_probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
        all_labels.extend(lbl.cpu().numpy().flatten())
    bin_preds = [1 if p > 0.5 else 0 for p in all_probs]
    return (total_loss / len(loader),
            accuracy_score(bin_preds, all_labels),
            roc_auc_score(all_labels, all_probs))


def run_training_session(model_class, train_loader_s, valid_loader_s, epochs, patience,
                          label="session", verbose=True):
    """Generic training session (differential LR, progressive unfreeze, label
    smoothing, gradient accumulation, CombinedLoss + ClinicalGateLoss + entropy
    reg when applicable, float32-safe). Used for every model in this notebook,
    so no variant gets an unfair advantage from a different training procedure."""
    set_global_seed(SEED)
    sess_model = model_class().to(DEVICE)
    for p in sess_model.backbone.shallow.parameters(): p.requires_grad = False
    for p in sess_model.backbone.deep.parameters():    p.requires_grad = False

    sess_labels = train_loader_s.dataset.get_labels()
    sess_pos = sum(sess_labels); sess_neg = len(sess_labels) - sess_pos
    sess_pos_weight = torch.tensor([sess_neg / max(sess_pos, 1)], dtype=torch.float32).to(DEVICE)

    sess_param_groups = [
        {'params': (list(sess_model.backbone.shallow.parameters()) +
                    list(sess_model.backbone.deep.parameters())),
         'lr': LR_BACKBONE, 'weight_decay': WD_BACKBONE},
        {'params': [p for n, p in sess_model.named_parameters()
                    if 'backbone.shallow' not in n and 'backbone.deep' not in n],
         'lr': LR_HEAD, 'weight_decay': WD_HEAD},
    ]
    sess_opt    = torch.optim.AdamW(sess_param_groups)
    sess_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(sess_opt, T_max=epochs, eta_min=1e-6)
    sess_scaler = torch.cuda.amp.GradScaler()
    sess_criterion = CombinedLoss(pos_weight=sess_pos_weight).to(DEVICE)
    val_criterion  = nn.BCEWithLogitsLoss().to(DEVICE)

    # Ratio-based progressive unfreeze: 15% / 30% of the epoch budget (thesis 3.5.2)
    unfreeze_partial = max(1, round(epochs * 0.15))
    unfreeze_full    = max(2, round(epochs * 0.30))#######30->35

    best_auc, best_state, patience_cnt = 0.0, None, 0
    hist = {'train_loss': [], 'train_auc': [], 'val_loss': [], 'val_auc': [],
            'gate_weights': []}   # <-- NEW: train_auc + gate_weights tracked

    for epoch in range(epochs):
        if epoch == unfreeze_partial:
            for p in sess_model.backbone.deep[-1].parameters():
                p.requires_grad = True
        if epoch == unfreeze_full:
            for p in sess_model.backbone.shallow.parameters(): p.requires_grad = True
            for p in sess_model.backbone.deep.parameters():    p.requires_grad = True

        t_loss, t_auc, gate_avg = train_epoch(sess_model, train_loader_s, sess_opt,
                                               sess_scaler, sess_criterion)   # <-- CHANGED
        v_loss, v_acc, v_auc = evaluate(sess_model, valid_loader_s, val_criterion)
        sess_sched.step()

        hist['train_loss'].append(t_loss)
        hist['train_auc'].append(t_auc)                 # <-- NEW
        hist['val_loss'].append(v_loss)
        hist['val_auc'].append(v_auc)
        hist['gate_weights'].append(gate_avg)            # <-- NEW

        if v_auc > best_auc:
            best_auc = v_auc
            best_state = {k: v.detach().clone().cpu() for k, v in sess_model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if verbose:
            # <-- NEW: gate weights formatted for the log line, e.g. gate=[sag=0.52, cor=0.31, ax=0.17]
            gate_str = ""
            if gate_avg is not None:
                gate_labels = ['sag', 'cor', 'ax'] if len(gate_avg) == 3 else [f'g{i}' for i in range(len(gate_avg))]
                gate_str = "  gate=[" + ", ".join(f"{n}={g:.3f}" for n, g in zip(gate_labels, gate_avg)) + "]"

            print(f"  [{label}] ep{epoch+1:>2}/{epochs} t_loss={t_loss:.4f} t_auc={t_auc:.4f} "
                  f"v_loss={v_loss:.4f} v_auc={v_auc:.4f}{gate_str}{'  *' if patience_cnt==0 else ''}")

        if patience_cnt >= patience:
            if verbose: print(f"  [{label}] early stop at epoch {epoch+1}")
            break

    sess_model.load_state_dict(best_state)
    sess_model.eval()
    probs_s, labels_s = [], []
    with torch.no_grad():
        for s_b, c_b, a_b, lbl_b in valid_loader_s:
            logits, _ = sess_model(s_b.to(DEVICE), c_b.to(DEVICE), a_b.to(DEVICE))
            probs_s.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            labels_s.extend(lbl_b.numpy().flatten())
    final_acc = accuracy_score([1 if p > 0.5 else 0 for p in probs_s], labels_s)
    final_f1  = f1_score(labels_s, [1 if p > 0.5 else 0 for p in probs_s], zero_division=0)

    # Return the trained model itself (not just its weights) so Grad-CAM
    # (Section 13) can run directly on the SAME model instance that produced
    # these numbers -- no more "which checkpoint is this?" ambiguity.
    result = {'model': sess_model, 'best_auc': best_auc, 'acc': final_acc, 'f1': final_f1,
              'history': hist, 'probs': probs_s, 'labels': labels_s}

    del sess_opt, sess_scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result


def hanley_mcneil_ci(auc, n_pos, n_neg, conf=0.95):
    """Analytical 95% CI for AUC (Hanley & McNeil 1982) -- thesis section 4.5."""
    q1 = auc / (2 - auc)
    q2 = (2 * auc**2) / (1 + auc)
    se2 = (auc*(1-auc) + (n_pos-1)*(q1-auc**2) + (n_neg-1)*(q2-auc**2)) / (n_pos*n_neg)
    se = math.sqrt(se2)
    z = 1.959963985
    return se, max(0.0, auc - z*se), min(1.0, auc + z*se)


print("[Training] train_epoch / evaluate / run_training_session / hanley_mcneil_ci ready.")

## 7. Train All 8 Variants (produces the numbers for thesis 4.3, 4.4, 4.7)

Proposed model + no-cross-attention ablation + 3 single-plane + Concat + Average + MRNet-style, all through the identical pipeline above. **This is the long-running cell** -- expect roughly the same order of magnitude as before (~1500 minutes total on a GTX 1650, per the previous run's own timing).

In [ ]:
all_variants = {
    'ACLCrossAttentionNet (proposed)':  ACLCrossAttentionNet,
    'Gated fusion, no cross-attention':  ACLNoCrossAttentionNet,   # = the ablation study, thesis 4.7
    'Sagittal-only':                     functools.partial(SinglePlaneNet, plane='sagittal'),
    'Coronal-only':                      functools.partial(SinglePlaneNet, plane='coronal'),
    'Axial-only':                        functools.partial(SinglePlaneNet, plane='axial'),
    'Concat fusion':                     ConcatFusionNet,
    'Average fusion':                    AverageFusionNet,
    'MRNet-style (Bien et al. 2018)':    MRNetStyleBaseline,
}

all_results = {}
run_t0 = time.time()

for name, cls in all_variants.items():
    print(f"\n{'='*60}\nTRAINING: {name}\n{'='*60}")
    tmp = cls()
    n_params = sum(p.numel() for p in tmp.parameters())
    del tmp

    result = run_training_session(cls, train_loader, valid_loader,
                                   epochs=EPOCHS, patience=PATIENCE, label=name)
    result['params'] = n_params
    all_results[name] = result
    print(f"  >>> {name}: AUC={result['best_auc']:.4f}  Acc={result['acc']:.4f}  "
          f"F1={result['f1']:.4f}  Params={n_params:,}")

run_elapsed = time.time() - run_t0
print(f"\n[All Variants] Total time: {run_elapsed/60:.1f} minutes "
      f"({run_elapsed/max(1,len(all_variants))/60:.1f} min/variant)")

# Keep a direct handle to the proposed model for Grad-CAM later (Section 13)
proposed_model = all_results['ACLCrossAttentionNet (proposed)']['model']


## 8. Results Table & Statistical Comparison (thesis Tables 4-1, 4-2, 4-4)

In [ ]:
comparison_rows = []
for name, r in all_results.items():
    n_pos = int(sum(r['labels'])); n_neg = len(r['labels']) - n_pos
    se, lo, hi = hanley_mcneil_ci(r['best_auc'], n_pos, n_neg)
    comparison_rows.append({
        'Model': name, 'AUC': r['best_auc'], 'AUC_CI_low': lo, 'AUC_CI_high': hi,
        'Accuracy': r['acc'], 'F1': r['f1'], 'Params': r['params']})

comparison_df = pd.DataFrame(comparison_rows).sort_values('AUC', ascending=False).reset_index(drop=True)
print(comparison_df.round(4).to_string(index=False))
comparison_df.to_csv(f"{FIG_DIR}/table_4-1_baseline_comparison.csv", index=False)
print(f"\n[Saved] table_4-1_baseline_comparison.csv -> Thesis Tables 4-1 / 4-2 / 4-4")

print("\nNote (thesis 4.5): with a 120-case validation set, treat AUC differences smaller than")
print("the overlap of these Hanley-McNeil intervals as not statistically distinguishable --")
print("report the intervals alongside the ranking, not the ranking alone.")

best_baseline_name = max(
    (n for n in all_results if n != 'ACLCrossAttentionNet (proposed)'),
    key=lambda k: all_results[k]['best_auc'])
margin = all_results['ACLCrossAttentionNet (proposed)']['best_auc'] - all_results[best_baseline_name]['best_auc']
print(f"\n[Summary] Best baseline: {best_baseline_name} (AUC={all_results[best_baseline_name]['best_auc']:.4f})")
print(f"[Summary] Proposed AUC margin over best baseline: {margin:+.4f}")

# ---- Ablation-specific detail table -> Table 4-3 -----------------------------
prop = all_results['ACLCrossAttentionNet (proposed)']
abl  = all_results['Gated fusion, no cross-attention']
n_pos_a = int(sum(prop['labels'])); n_neg_a = len(prop['labels']) - n_pos_a
_, plo, phi = hanley_mcneil_ci(prop['best_auc'], n_pos_a, n_neg_a)
_, alo, ahi = hanley_mcneil_ci(abl['best_auc'], n_pos_a, n_neg_a)
ablation_table = pd.DataFrame([
    {'Metric': 'AUC', 'With Cross-Attention': prop['best_auc'], 'Without Cross-Attention': abl['best_auc'],
     'Delta': prop['best_auc'] - abl['best_auc']},
    {'Metric': 'AUC 95% CI', 'With Cross-Attention': f"[{plo:.4f}, {phi:.4f}]",
     'Without Cross-Attention': f"[{alo:.4f}, {ahi:.4f}]", 'Delta': 'overlapping' if plo < ahi else 'separated'},
    {'Metric': 'Accuracy', 'With Cross-Attention': prop['acc'], 'Without Cross-Attention': abl['acc'],
     'Delta': prop['acc'] - abl['acc']},
    {'Metric': 'F1', 'With Cross-Attention': prop['f1'], 'Without Cross-Attention': abl['f1'],
     'Delta': prop['f1'] - abl['f1']},
    {'Metric': 'Parameters', 'With Cross-Attention': prop['params'], 'Without Cross-Attention': abl['params'],
     'Delta': prop['params'] - abl['params']},
])
print("\n" + ablation_table.to_string(index=False))
ablation_table.to_csv(f"{FIG_DIR}/table_4-3_ablation_detail.csv", index=False)
print(f"\n[Saved] table_4-3_ablation_detail.csv -> Thesis Table 4-3")


## 9. Visualizations for Chapter 4 (Figures 4-1, 4-2, 4-3, 4-8)

In [ ]:
# ---- Figure 4-1: proposed model's own training trajectory -------------------
prop_hist = all_results['ACLCrossAttentionNet (proposed)']['history']
epochs_ran = list(range(1, len(prop_hist['val_auc']) + 1))
best_ep = int(np.argmax(prop_hist['val_auc'])) + 1
best_auc_val = max(prop_hist['val_auc'])
unfreeze_partial_ep = max(1, round(EPOCHS * 0.15)) + 1   # 1-indexed
unfreeze_full_ep    = max(2, round(EPOCHS * 0.30)) + 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax1.plot(epochs_ran, prop_hist['val_auc'], color='seagreen', lw=2.2, marker='o', markersize=4, label='Validation AUC')
ax1.axhline(y=best_auc_val, color='seagreen', ls='--', alpha=0.4)
ax1.scatter([best_ep], [best_auc_val], color='seagreen', s=90, zorder=5, edgecolor='white', linewidth=1.5,
            label=f'Best checkpoint (epoch {best_ep}, AUC={best_auc_val:.4f})')
ax1.axvline(x=unfreeze_partial_ep, color='#B8873A', ls=':', alpha=0.7)
ax1.axvline(x=unfreeze_full_ep, color='#C0392B', ls=':', alpha=0.7)
ax1.text(unfreeze_partial_ep, min(prop_hist['val_auc'])*0.97, ' partial\n unfreeze', fontsize=8.5, color='#B8873A')
ax1.text(unfreeze_full_ep, min(prop_hist['val_auc'])*0.97, ' full\n unfreeze', fontsize=8.5, color='#C0392B')
ax1.set_ylabel("Validation AUC")
ax1.set_title(f"Proposed Model -- Training Trajectory (this run, {EPOCHS}-epoch budget, patience={PATIENCE})")
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, prop_hist['train_loss'], color='steelblue', lw=2, label='Train Loss')
ax2.plot(epochs_ran, prop_hist['val_loss'], color='darkorange', lw=2, label='Validation Loss')
ax2.axvline(x=unfreeze_partial_ep, color='#B8873A', ls=':', alpha=0.7)
ax2.axvline(x=unfreeze_full_ep, color='#C0392B', ls=':', alpha=0.7)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (L_total)")
ax2.legend(loc='upper right', fontsize=9); ax2.grid(True, alpha=0.3)
ax2.set_xticks(epochs_ran)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-1_proposed_training_curve.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-1_proposed_training_curve.png -> Thesis Figure 4-1")
print(f"[Numbers for thesis 4.3 text] best epoch={best_ep}, best AUC={best_auc_val:.4f}, "
      f"total epochs run={len(epochs_ran)}")


In [ ]:
# ---- Figure 4-2: AUC / Accuracy / F1 grouped bar comparison ------------------
sorted_names = comparison_df['Model'].tolist()
auc_v = comparison_df['AUC'].tolist()
acc_v = comparison_df['Accuracy'].tolist()
f1_v  = comparison_df['F1'].tolist()

y = np.arange(len(sorted_names)); h = 0.25
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(y + h, auc_v, height=h, label="AUC", color="#2a78d6")
ax.barh(y,     acc_v, height=h, label="Accuracy", color="#eb6834")
ax.barh(y - h, f1_v,  height=h, label="F1", color="#1baf7a")
ax.set_yticks(y); ax.set_yticklabels(sorted_names, fontsize=9)
ax.set_xlim(0, 1); ax.set_xlabel("Score")
ax.set_title("Proposed Model vs Baselines -- AUC / Accuracy / F1")
ax.invert_yaxis(); ax.legend(loc="lower right"); ax.grid(axis="x", linestyle="--", alpha=0.4)
for i, (a, c, f) in enumerate(zip(auc_v, acc_v, f1_v)):
    ax.text(a + 0.01, i + h, f"{a:.3f}", va="center", fontsize=8)
    ax.text(c + 0.01, i,     f"{c:.3f}", va="center", fontsize=8)
    ax.text(f + 0.01, i - h, f"{f:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-2_metrics_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-2_metrics_comparison.png -> Thesis Figure 4-2")


In [ ]:
# ---- Figure 4-3: AUC with Hanley-McNeil 95% CI error bars ---------------------
names_ci, aucs_ci, err_lo, err_hi = [], [], [], []
for name, r in all_results.items():
    n_pos = int(sum(r['labels'])); n_neg = len(r['labels']) - n_pos
    se, lo, hi = hanley_mcneil_ci(r['best_auc'], n_pos, n_neg)
    names_ci.append(name); aucs_ci.append(r['best_auc'])
    err_lo.append(r['best_auc'] - lo); err_hi.append(hi - r['best_auc'])

order = np.argsort(aucs_ci)[::-1]
names_ci = [names_ci[i] for i in order]; aucs_ci = [aucs_ci[i] for i in order]
err_lo = [err_lo[i] for i in order]; err_hi = [err_hi[i] for i in order]
colors_ci = ['seagreen' if n.startswith('ACLCrossAttentionNet') else 'steelblue' for n in names_ci]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(names_ci, aucs_ci, xerr=[err_lo, err_hi], color=colors_ci, alpha=0.85, capsize=3)
for i, v in enumerate(aucs_ci):
    ax.text(v + 0.015, i, f"{v:.3f}", va='center', fontsize=9)
ax.set_xlabel("Validation AUC (95% CI, Hanley-McNeil)")
ax.set_title(f"AUC with 95% Confidence Intervals -- Every Model, Identical Pipeline\n"
             f"({EPOCHS}-epoch budget, same split, same loss)")
ax.set_xlim(0.3, 1.05); ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-3_auc_confidence_intervals.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-3_auc_confidence_intervals.png -> Thesis Figure 4-3")

# Overlap check -> the key honest finding of thesis section 4.5 / 5.1
prop_lo = aucs_ci[names_ci.index('ACLCrossAttentionNet (proposed)')] - err_lo[names_ci.index('ACLCrossAttentionNet (proposed)')]
print(f"\n[Check] Proposed model CI lower bound: {prop_lo:.4f}")
for n, a, eh in zip(names_ci, aucs_ci, err_hi):
    if n == 'ACLCrossAttentionNet (proposed)': continue
    overlaps = (a + eh) > prop_lo
    print(f"  vs {n}: CI upper={a+eh:.4f} -> {'OVERLAPS' if overlaps else 'separated'} with proposed's lower bound")


In [ ]:
# ---- Figure 4-8: parameter count comparison ------------------------------------
params_v = comparison_df.set_index('Model').loc[sorted_names, 'Params'].tolist()
colors_p = ["#2a78d6" if n == "ACLCrossAttentionNet (proposed)" else "#c3c2b7" for n in sorted_names]
fig2, ax2b = plt.subplots(figsize=(9, 5))
ax2b.barh(sorted_names, [p/1e6 for p in params_v], color=colors_p)
ax2b.set_xlabel("Parameters (millions)")
ax2b.set_title("Model Size Comparison")
ax2b.invert_yaxis()
for i, p in enumerate(params_v):
    ax2b.text(p/1e6 + 0.1, i, f"{p:,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-8_params_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-8_params_comparison.png -> Thesis Figure 4-8")


## 10. Threshold Optimization (thesis section 4.2 method, closes the open bracket in 4.3)


In [ ]:
proposed_probs  = all_results['ACLCrossAttentionNet (proposed)']['probs']
proposed_labels = all_results['ACLCrossAttentionNet (proposed)']['labels']

thresholds = [i/100 for i in range(10, 70, 2)]
thr_results = []
for thr in thresholds:
    preds_thr = [1 if p > thr else 0 for p in proposed_probs]
    f1  = f1_score(proposed_labels, preds_thr, zero_division=0)
    tp  = sum(1 for p, l in zip(preds_thr, proposed_labels) if p == 1 and l == 1)
    rec = tp / max(sum(proposed_labels), 1)
    pre = tp / max(sum(preds_thr), 1)
    thr_results.append((thr, f1, pre, rec))

best_thr, best_f1, best_pre, best_rec = max(thr_results, key=lambda x: x[1])
print(f"[THRESHOLD OPTIMIZATION -- official run]")
print(f"Optimal threshold : {best_thr:.2f}")
print(f"At this threshold : Precision={best_pre:.3f}  Recall={best_rec:.3f}  F1={best_f1:.3f}")

opt_preds = [1 if p > best_thr else 0 for p in proposed_probs]
opt_acc = accuracy_score(opt_preds, proposed_labels)
print(f"Accuracy at optimal threshold: {opt_acc:.4f}")
print()
print(">>>THIS SENTENCE INTO THESIS SECTION 4.3 ( open bracket) <<<")
print(f'"در آستانه‌ی بهینه‌ی {best_thr:.2f}، مدل پیشنهادی به دقت {opt_acc*100:.2f}٪ و F1 برابر با '
      f'{best_f1:.3f} دست یافت."')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in thr_results], [r[2] for r in thr_results], label='Precision (ACL)', color='steelblue')
ax.plot([r[0] for r in thr_results], [r[3] for r in thr_results], label='Recall (ACL)', color='darkorange')
ax.plot([r[0] for r in thr_results], [r[1] for r in thr_results], label='F1 (ACL)', color='seagreen', lw=2)
ax.axvline(x=best_thr, color='red', ls='--', alpha=0.6, label=f'Optimal thr={best_thr:.2f}')
ax.axvline(x=0.50, color='gray', ls=':', alpha=0.5, label='Default thr=0.50')
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs Threshold (ACL Tear class, official run)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-x_threshold_search_official.png", dpi=200, bbox_inches='tight')
plt.show()
print("\n[Saved] fig_4-x_threshold_search_official.png -> Thesis Figure 4-1 (section 4.2, as previously agreed)")

opt_cm = confusion_matrix(proposed_labels, opt_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(opt_cm, annot=True, fmt='d', cmap='Greens', xticklabels=["Normal","Torn ACL"], yticklabels=["Normal","Torn ACL"])
plt.title(f"Confusion Matrix -- Optimal Threshold ({best_thr:.2f}), Official Run")
plt.ylabel("True Label"); plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-x_confusion_matrix_optimal.png", dpi=200, bbox_inches='tight')
plt.show()


## 11. Temperature Calibration (probability reliability check)

In [ ]:
class TemperatureScaler(nn.Module):
    """Post-hoc probability calibration via learned temperature T.
    calibrated_prob = sigmoid(logit / T). Does NOT modify model weights or AUC
    (AUC is rank-order invariant) -- only affects probability reliability."""
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1, device=DEVICE))

    def forward(self, logits):
        return logits / self.temperature.clamp(min=0.3, max=5.0)


ts_scaler = TemperatureScaler().to(DEVICE)
ts_opt = torch.optim.LBFGS([ts_scaler.temperature], lr=0.01, max_iter=100)
nll = nn.BCEWithLogitsLoss()

proposed_model.eval()
logits_all, labels_all = [], []
with torch.no_grad():
    for s_b, c_b, a_b, lbl_b in valid_loader:
        out, _ = proposed_model(s_b.to(DEVICE), c_b.to(DEVICE), a_b.to(DEVICE))
        logits_all.append(out.float().cpu())
        labels_all.append(lbl_b)
logits_all = torch.cat(logits_all).to(DEVICE)
labels_all = torch.cat(labels_all).to(DEVICE)

def cal_closure():
    ts_opt.zero_grad()
    loss = nll(ts_scaler(logits_all), labels_all)
    loss.backward()
    return loss

ts_opt.step(cal_closure)
with torch.no_grad():
    cal_probs = torch.sigmoid(ts_scaler(logits_all)).cpu().numpy()
cal_auc = roc_auc_score(labels_all.cpu().numpy(), cal_probs)
print(f"[Calibration] Optimal temperature T = {ts_scaler.temperature.item():.4f}")
print(f"[Calibration] Uncalibrated AUC = {all_results['ACLCrossAttentionNet (proposed)']['best_auc']:.4f}")
print(f"[Calibration] Calibrated   AUC = {cal_auc:.4f}  (should match -- AUC is rank-order invariant)")


## 12. Interpretability: Grad-CAM + Slice-Attention Weights (thesis 4.6)

Runs on `proposed_model` -- the exact model trained in Section 7 of **this** notebook, so case numbers
and probabilities here are internally consistent with every other number in this run.

A standard hook-based Grad-CAM library is not safe on this architecture: `model.backbone` is called
three times per forward pass (once per plane), so a naive forward-hook fires three times and silently
captures whichever call lands last. The extractor below manually reconstructs the forward pass and
retains gradients only on the target plane's own backbone output, avoiding that ambiguity (thesis 4.6).

In [ ]:
class GradCAMExtractor:
    """Manual Grad-CAM for ACLCrossAttentionNet's shared multi-view backbone (thesis 4.6)."""
    def __init__(self, model):
        self.model = model

    def generate(self, sag, cor, ax, plane='sagittal'):
        self.model.eval()
        b, s, h, w = sag.shape
        inputs = {'sagittal': sag, 'coronal': cor, 'axial': ax}
        target_input = inputs[plane].clone().to(DEVICE).requires_grad_(True)
        sag_, cor_, ax_ = sag.to(DEVICE), cor.to(DEVICE), ax.to(DEVICE)

        feat = self.model.backbone(target_input)
        feat.retain_grad()
        fs = feat if plane == 'sagittal' else self.model.backbone(sag_)
        fc = feat if plane == 'coronal'  else self.model.backbone(cor_)
        fa = feat if plane == 'axial'    else self.model.backbone(ax_)

        ts = self.model.tokenizer(fs, b, s); tc = self.model.tokenizer(fc, b, s); ta = self.model.tokenizer(fa, b, s)
        ts = self.model.slice_attn(ts); tc = self.model.slice_attn(tc); ta = self.model.slice_attn(ta)
        ts, tc, ta = self.model.cross_block_1(ts, tc, ta)
        ts, tc, ta = self.model.cross_block_2(ts, tc, ta)
        fused, gate_w = self.model.gated_fusion(ts, tc, ta)
        ctx = self.model.context_enc(fused)
        vec = ctx.mean(dim=1)
        logit = self.model.classifier(vec)

        self.model.zero_grad()
        logit.squeeze().backward()

        grads = feat.grad
        acts  = feat.detach()
        weights = grads.mean(dim=(2, 3), keepdim=True)
        raw_cam = F.relu((weights * acts).sum(dim=1)).view(b * s, 7, 7)
        cam_up = F.interpolate(raw_cam.unsqueeze(1), size=(h, w), mode='bilinear', align_corners=False).squeeze(1)
        cam_up = cam_up.detach().cpu().numpy()

        for i in range(cam_up.shape[0]):
            c = cam_up[i]
            rng = float(c.max() - c.min())
            cam_up[i] = (c - c.min()) / rng if rng > 0 else np.zeros_like(c)

        return cam_up, torch.sigmoid(logit).item(), gate_w.detach().cpu().numpy()[0]


def get_slice_attention_weights(model, sag, cor, ax):
    """Reads the slice-attention MLP's raw scores via a read-only forward hook
    (thesis 4.6, second interpretability signal -- not gradient-derived)."""
    captured = {}
    def hook(module, inp, out):
        captured['logits'] = out.detach()
    h = model.slice_attn.mlp.register_forward_hook(hook)

    model.eval()
    with torch.no_grad():
        b, s, hh, ww = sag.shape
        weights_per_plane = {}
        for name, view in [('sagittal', sag), ('coronal', cor), ('axial', ax)]:
            f = model.backbone(view.to(DEVICE))
            t_raw = model.tokenizer(f, b, s)
            _ = model.slice_attn(t_raw)
            w = F.softmax(captured['logits'], dim=1)
            w = w.squeeze(-1).squeeze(0).mean(dim=-1)
            weights_per_plane[name] = w.cpu().numpy()
    h.remove()
    return weights_per_plane['sagittal'], weights_per_plane['coronal'], weights_per_plane['axial']


gradcam = GradCAMExtractor(proposed_model)
print("[Grad-CAM] Extractor ready (per-plane isolated, runs on this run's proposed_model).")


In [ ]:
# ---- Select representative cases using THIS run's optimal threshold (Section 10) ----
def get_case(idx):
    sag, cor, ax, lbl = valid_ds[idx]
    return sag.unsqueeze(0), cor.unsqueeze(0), ax.unsqueeze(0), lbl.item()

probs_arr  = np.array(proposed_probs)
labels_arr = np.array(proposed_labels)

tp_mask = (labels_arr == 1) & (probs_arr > best_thr)
fn_mask = (labels_arr == 1) & (probs_arr <= best_thr)
tn_mask = (labels_arr == 0) & (probs_arr <= best_thr)

tp_idx_sorted = np.where(tp_mask)[0][np.argsort(-probs_arr[np.where(tp_mask)[0]])]
fn_idx_sorted = np.where(fn_mask)[0]
tn_idx_sorted = np.where(tn_mask)[0][np.argsort(probs_arr[np.where(tn_mask)[0]])]

print(f"[Grad-CAM] True Positives : {len(tp_idx_sorted)}")
print(f"[Grad-CAM] False Negatives: {len(fn_idx_sorted)}  (used for the honest failure-case figure)")
print(f"[Grad-CAM] True Negatives : {len(tn_idx_sorted)}")


In [ ]:
# ---- Figure 4-4: single case, all 3 planes -------------------------------------
sag_t, cor_t, ax_t, true_lbl = get_case(66)
mid = MAX_SLICES // 2

fig, axs = plt.subplots(2, 3, figsize=(13, 8))
plane_inputs = {'sagittal': sag_t, 'coronal': cor_t, 'axial': ax_t}
prob_demo = None
for col, plane in enumerate(['sagittal', 'coronal', 'axial']):
    cam, prob_demo, gw = gradcam.generate(sag_t, cor_t, ax_t, plane=plane)
    raw_img = plane_inputs[plane][0, mid].numpy()
    raw_img = (raw_img - raw_img.min()) / (raw_img.max() - raw_img.min() + 1e-8)
    axs[0, col].imshow(raw_img, cmap='gray'); axs[0, col].set_title(f"{plane.capitalize()} (slice {mid})"); axs[0, col].axis('off')
    axs[1, col].imshow(raw_img, cmap='gray'); axs[1, col].imshow(cam[mid], cmap='jet', alpha=0.45)
    axs[1, col].set_title(f"Grad-CAM (gate={gw[col]:.2f})"); axs[1, col].axis('off')

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-4_gradcam_3plane_case.png", dpi=200, bbox_inches='tight')
plt.show()
print(f"[Saved] fig_4-4_gradcam_3plane_case.png -> Thesis Figure 4-4 ")


In [ ]:
# ---- Figure 4-5: multiple correctly-detected cases, sagittal plane -----------
n_show = min(6, len(tp_idx_sorted))
fig, axs = plt.subplots(2, 3, figsize=(13, 8))
for k in range(n_show):
    idx = int(tp_idx_sorted[k])
    sag_k, cor_k, ax_k, _ = get_case(idx)
    cam_k, prob_k, gw_k = gradcam.generate(sag_k, cor_k, ax_k, plane='sagittal')
    raw_k = sag_k[0, mid].numpy()
    raw_k = (raw_k - raw_k.min()) / (raw_k.max() - raw_k.min() + 1e-8)
    r, c = divmod(k, 3)
    axs[r, c].imshow(raw_k, cmap='gray'); axs[r, c].imshow(cam_k[mid], cmap='jet', alpha=0.45)
    axs[r, c].set_title(f"Case #{idx}  p={prob_k:.2f}", fontsize=10); axs[r, c].axis('off')
for k in range(n_show, 6):
    r, c = divmod(k, 3); axs[r, c].axis('off')
plt.suptitle("Sagittal Grad-CAM Across Multiple Correctly-Detected ACL Tears", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-5_gradcam_multicase_grid.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-5_gradcam_multicase_grid.png -> Thesis Figure 4-5")


In [ ]:
# ---- Figure 4-6: honest failure case (False Negative) -------------------------
if len(fn_idx_sorted) > 0:
    fn_idx = int(fn_idx_sorted[0])
    sag_f, cor_f, ax_f, _ = get_case(fn_idx)
    cam_f, prob_f, gw_f = gradcam.generate(sag_f, cor_f, ax_f, plane='sagittal')
    raw_f = sag_f[0, mid].numpy()
    raw_f = (raw_f - raw_f.min()) / (raw_f.max() - raw_f.min() + 1e-8)

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(raw_f, cmap='gray'); axs[0].set_title(f"Case #{fn_idx} -- True label: Torn ACL"); axs[0].axis('off')
    axs[1].imshow(raw_f, cmap='gray'); axs[1].imshow(cam_f[mid], cmap='jet', alpha=0.45)
    axs[1].set_title(f"Model predicted p={prob_f:.3f} (missed, thr={best_thr:.2f})"); axs[1].axis('off')
    plt.suptitle("Failure Case -- False Negative (honest limitation discussion, thesis 4.6)", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/fig_4-6_gradcam_failure_case.png", dpi=200, bbox_inches='tight')
    plt.show()
    print(f"[Saved] fig_4-6_gradcam_failure_case.png -> Thesis Figure 4-6 (this run: case #{fn_idx}, p={prob_f:.3f})")
else:
    print("[Grad-CAM] No false negatives at the optimal threshold in this run -- skipping Figure 4-6.")
    print("[Grad-CAM] NOTE: if this differs from the thesis text (which describes a case #44), the")
    print("[Grad-CAM] failure-case paragraph in section 4.6 needs updating to match this run's actual result.")


In [ ]:
# ---- Figure 4-7: learned slice-attention weights, representative case ---------
w_sag, w_cor, w_ax = get_slice_attention_weights(proposed_model, sag_t, cor_t, ax_t)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(MAX_SLICES); width = 0.25
ax.bar(x - width, w_sag, width, label='Sagittal', color='steelblue')
ax.bar(x,          w_cor, width, label='Coronal',  color='darkorange')
ax.bar(x + width,  w_ax,  width, label='Axial',    color='seagreen')
ax.set_xlabel("Slice index"); ax.set_ylabel("Attention weight")
ax.set_title(f"Learned Slice-Attention Weights -- Case #{demo_idx}")
ax.set_xticks(x); ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-7_slice_attention_weights.png", dpi=200, bbox_inches='tight')
plt.show()
print(f"[Saved] fig_4-7_slice_attention_weights.png -> Thesis Figure 4-7 (this run: case #{demo_idx})")

print("\n[Appendix material] For a multi-slice montage of all 16 sagittal slices (suggested for an")
print("appendix rather than the main text, per the earlier size discussion), run:")
print(">>> cam_full, _, _ = gradcam.generate(sag_t, cor_t, ax_t, plane='sagittal')")
print(">>> # then plot cam_full[0..15] in a 4x4 grid, same pattern as Figures 4-4/4-5 above.")


## 13. Final Consolidated Summary (thesis section 4.8, Table 4-4)

In [ ]:
print("="*70)
print("FINAL SUMMARY -- everything needed to complete the thesis text")
print("="*70)
print(f"\n1. Chapter 4.3 -- proposed model:")
print(f"   Best epoch: {best_ep} / {len(epochs_ran)} run | Best AUC: {best_auc_val:.4f}")
print(f"   Default threshold (0.50): Acc={all_results['ACLCrossAttentionNet (proposed)']['acc']:.4f}  "
      f"F1={all_results['ACLCrossAttentionNet (proposed)']['f1']:.4f}")
print(f"   Optimal threshold ({best_thr:.2f}): Acc={opt_acc:.4f}  F1={best_f1:.3f}   <- fills the open bracket")

print(f"\n2. Chapter 4.4/4.8 -- full comparison table:")
print(comparison_df.round(4).to_string(index=False))

print(f"\n3. Chapter 4.7 -- ablation:")
print(ablation_table.to_string(index=False))

print(f"\n4. All output files saved to ./{FIG_DIR}/ :")
for fn in sorted(os.listdir(FIG_DIR)):
    print(f"   - {fn}")

print("\n5. Reminders:")
print("   - K-Fold cross-validation was intentionally NOT run here (thesis 5.4, future work).")
print("   - If the Grad-CAM case numbers above differ from what thesis 4.6 currently says")
print("     (case #63 / #44 from the older run), update those paragraphs to match THIS run's")
print("     case numbers and probabilities, printed next to each figure above.")
print("   - Re-run comparison_df/ablation_table cells any time to regenerate the CSVs.")


In [ ]:
from sklearn.metrics import precision_score, recall_score, matthews_corrcoef, balanced_accuracy_score

extended_rows = []
for name, r in all_results.items():
    probs  = r['probs']
    labels = r['labels']
    preds  = [1 if p > 0.5 else 0 for p in probs]   # همون آستانه‌ی 0.5 که در comparison_df استفاده شده

    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)          # = Sensitivity

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    mcc     = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)

    extended_rows.append({
        'Model': name,
        'Precision': prec,
        'Recall (Sensitivity)': rec,
        'Specificity': specificity,
        'Balanced Acc': bal_acc,
        'MCC': mcc,
    })

extended_df = pd.DataFrame(extended_rows)
comparison_df_full = comparison_df.merge(extended_df, on='Model').sort_values('AUC', ascending=False).reset_index(drop=True)

print(comparison_df_full.round(4).to_string(index=False))
comparison_df_full.to_csv(f"{FIG_DIR}/table_4-1_baseline_comparison_extended.csv", index=False)
print(f"\n[Saved] table_4-1_baseline_comparison_extended.csv")

In [ ]:
# ---- Figure 4-2: AUC / Accuracy / F1 / Precision / Recall / Specificity comparison ----
plot_df = comparison_df_full  # already sorted by AUC descending
sorted_names = plot_df['Model'].tolist()
y = np.arange(len(sorted_names))
h = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- left panel: AUC / Accuracy / F1 (unchanged) ---
ax = axes[0]
auc_v = plot_df['AUC'].tolist()
acc_v = plot_df['Accuracy'].tolist()
f1_v  = plot_df['F1'].tolist()
ax.barh(y + h, auc_v, height=h, label="AUC", color="#2a78d6")
ax.barh(y,     acc_v, height=h, label="Accuracy", color="#eb6834")
ax.barh(y - h, f1_v,  height=h, label="F1", color="#1baf7a")
ax.set_yticks(y); ax.set_yticklabels(sorted_names, fontsize=9)
ax.set_xlim(0, 1); ax.set_xlabel("Score")
ax.set_title("AUC / Accuracy / F1")
ax.invert_yaxis(); ax.legend(loc="lower right"); ax.grid(axis="x", linestyle="--", alpha=0.4)
for i, (a, c, f) in enumerate(zip(auc_v, acc_v, f1_v)):
    ax.text(a + 0.01, i + h, f"{a:.3f}", va="center", fontsize=7.5)
    ax.text(c + 0.01, i,     f"{c:.3f}", va="center", fontsize=7.5)
    ax.text(f + 0.01, i - h, f"{f:.3f}", va="center", fontsize=7.5)

# --- right panel: Precision / Recall (Sensitivity) / Specificity (new) ---
ax2 = axes[1]
prec_v = plot_df['Precision'].tolist()
rec_v  = plot_df['Recall (Sensitivity)'].tolist()
spec_v = plot_df['Specificity'].tolist()
ax2.barh(y + h, prec_v, height=h, label="Precision", color="#8e44ad")
ax2.barh(y,     rec_v,  height=h, label="Recall (Sensitivity)", color="#e67e22")
ax2.barh(y - h, spec_v, height=h, label="Specificity", color="#16a085")
ax2.set_xlim(0, 1); ax2.set_xlabel("Score")
ax2.set_title("Precision / Recall / Specificity")
ax2.legend(loc="lower right"); ax2.grid(axis="x", linestyle="--", alpha=0.4)
for i, (p, r, s) in enumerate(zip(prec_v, rec_v, spec_v)):
    ax2.text(p + 0.01, i + h, f"{p:.3f}", va="center", fontsize=7.5)
    ax2.text(r + 0.01, i,     f"{r:.3f}", va="center", fontsize=7.5)
    ax2.text(s + 0.01, i - h, f"{s:.3f}", va="center", fontsize=7.5)

fig.suptitle("Proposed Model vs Baselines -- Full Metric Comparison", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-2_metrics_comparison_extended.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-2_metrics_comparison_extended.png -> Thesis Figure 4-2 (extended)")

In [ ]:
# ---- Figure 4-9: Confusion Matrix grid, all 8 variants (threshold = 0.5) ----
model_names = comparison_df_full['Model'].tolist()   # sorted by AUC, same order as the table
n_models = len(model_names)
n_cols = 4
n_rows = math.ceil(n_models / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, name in enumerate(model_names):
    r = all_results[name]
    preds = [1 if p > 0.5 else 0 for p in r['probs']]
    cm = confusion_matrix(r['labels'], preds)

    ax = axes[i]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=["Normal", "Torn ACL"], yticklabels=["Normal", "Torn ACL"],
                cbar=False, ax=ax, annot_kws={"size": 11})
    auc_val = comparison_df_full.loc[comparison_df_full['Model'] == name, 'AUC'].values[0]
    ax.set_title(f"{name}\nAUC={auc_val:.4f}", fontsize=9)
    ax.set_ylabel("True" if i % n_cols == 0 else "")
    ax.set_xlabel("Predicted" if i >= (n_rows - 1) * n_cols else "")

# hide any unused subplot slots
for j in range(n_models, len(axes)):
    axes[j].axis('off')

fig.suptitle("Confusion Matrices -- All Variants (threshold = 0.50)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-9_confusion_matrix_grid.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-9_confusion_matrix_grid.png -> Thesis Figure 4-9 (new)")

In [ ]:
# ---- Figure 4-10: MCC comparison, all 8 variants (threshold = 0.5) ----
plot_df = comparison_df_full.sort_values('MCC', ascending=True)   # ascending so best MCC ends up on top after invert
names_mcc = plot_df['Model'].tolist()
mcc_v     = plot_df['MCC'].tolist()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#2a78d6" if name == "ACLCrossAttentionNet (proposed)" else "#95a5a6" for name in names_mcc]
bars = ax.barh(names_mcc, mcc_v, color=colors)

ax.axvline(x=0, color='black', lw=0.8)
ax.set_xlim(-1, 1)
ax.set_xlabel("Matthews Correlation Coefficient (MCC)")
ax.set_title("Proposed Model vs Baselines -- MCC (threshold = 0.50)")
ax.grid(axis="x", linestyle="--", alpha=0.4)

for bar, v in zip(bars, mcc_v):
    x_pos = v + 0.02 if v >= 0 else v - 0.02
    ha = 'left' if v >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f"{v:.3f}", va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-10_mcc_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-10_mcc_comparison.png -> Thesis Figure 4-10 (new)")

In [ ]:
# ---- Figure 4-11: ROC Curves, all 8 variants overlaid ----
plot_order = comparison_df_full['Model'].tolist()   # sorted by AUC descending, consistent with the table

fig, ax = plt.subplots(figsize=(8, 7))
color_cycle = plt.cm.tab10(np.linspace(0, 1, len(plot_order)))

for name, color in zip(plot_order, color_cycle):
    r = all_results[name]
    fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
    roc_auc_val = sk_auc(fpr, tpr)

    lw = 2.5 if name == "ACLCrossAttentionNet (proposed)" else 1.3
    ls = '-' if name == "ACLCrossAttentionNet (proposed)" else '--'
    ax.plot(fpr, tpr, color=color, lw=lw, ls=ls,
            label=f"{name} (AUC={roc_auc_val:.3f})")

ax.plot([0, 1], [0, 1], color='gray', lw=1, ls=':', label='Chance (AUC=0.500)')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 - Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity)")
ax.set_title("ROC Curves -- Proposed Model vs Baselines")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-11_roc_curves_all_variants.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-11_roc_curves_all_variants.png -> Thesis Figure 4-11 (new)")

In [ ]:
# ---- Figure 4-12: Precision-Recall Curves, all 8 variants overlaid ----
from sklearn.metrics import precision_recall_curve, average_precision_score

plot_order = comparison_df_full['Model'].tolist()   # sorted by AUC descending, consistent with earlier figures
n_pos_total = int(sum(all_results['ACLCrossAttentionNet (proposed)']['labels']))
n_total     = len(all_results['ACLCrossAttentionNet (proposed)']['labels'])
baseline_prevalence = n_pos_total / n_total

fig, ax = plt.subplots(figsize=(8, 7))
color_cycle = plt.cm.tab10(np.linspace(0, 1, len(plot_order)))

for name, color in zip(plot_order, color_cycle):
    r = all_results[name]
    prec, rec, _ = precision_recall_curve(r['labels'], r['probs'])
    ap = average_precision_score(r['labels'], r['probs'])

    lw = 2.5 if name == "ACLCrossAttentionNet (proposed)" else 1.3
    ls = '-' if name == "ACLCrossAttentionNet (proposed)" else '--'
    ax.plot(rec, prec, color=color, lw=lw, ls=ls,
            label=f"{name} (AP={ap:.3f})")

ax.axhline(y=baseline_prevalence, color='gray', lw=1, ls=':',
           label=f'Chance (prevalence={baseline_prevalence:.3f})')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel("Recall (Sensitivity)")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves -- Proposed Model vs Baselines")
ax.legend(loc="lower left", fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_4-13_pr_curves_all_variants.png", dpi=200, bbox_inches='tight')
plt.show()
print("[Saved] fig_4-13_pr_curves_all_variants.png -> Thesis Figure 4-13 (new)")

# ---- also fold Average Precision into the main comparison table ----
ap_rows = {name: average_precision_score(all_results[name]['labels'], all_results[name]['probs'])
           for name in all_results}
comparison_df_full['Avg Precision (PR-AUC)'] = comparison_df_full['Model'].map(ap_rows)
print(comparison_df_full[['Model', 'AUC', 'Avg Precision (PR-AUC)']].round(4).to_string(index=False))
comparison_df_full.to_csv(f"{FIG_DIR}/table_4-1_baseline_comparison_extended.csv", index=False)
print(f"\n[Saved] table_4-1_baseline_comparison_extended.csv (updated with PR-AUC)")

In [ ]:
# ---- Classification Report (per-class) for all 8 variants, threshold = 0.5 ----
print("="*70)
print("CLASSIFICATION REPORTS -- threshold = 0.50")
print("="*70)
for name in comparison_df_full['Model'].tolist():   # sorted by AUC descending
    r = all_results[name]
    preds = [1 if p > 0.5 else 0 for p in r['probs']]
    print(f"\n--- {name} ---")
    print(classification_report(r['labels'], preds, target_names=["Normal", "Torn ACL"],
                                 zero_division=0, digits=3))

# consolidated text file for the thesis appendix
with open(f"{FIG_DIR}/classification_reports_all_variants.txt", "w", encoding="utf-8") as f:
    for name in comparison_df_full['Model'].tolist():
        r = all_results[name]
        preds = [1 if p > 0.5 else 0 for p in r['probs']]
        f.write(f"=== {name} ===\n")
        f.write(classification_report(r['labels'], preds, target_names=["Normal", "Torn ACL"],
                                       zero_division=0, digits=3))
        f.write("\n\n")
print(f"\n[Saved] classification_reports_all_variants.txt")

In [ ]:
# ---- Sensitivity at fixed Specificity (90% and 95%) -- clinical operating points ----
target_specificities = [0.90, 0.95]

sens_rows = []
for name in comparison_df_full['Model'].tolist():
    r = all_results[name]
    fpr, tpr, thr = roc_curve(r['labels'], r['probs'])
    specificity_arr = 1 - fpr

    row = {'Model': name}
    for target_spec in target_specificities:
        # بین تمام آستانه‌هایی که specificity >= هدف رو تامین می‌کنن، بیشترین sensitivity ممکن
        mask = specificity_arr >= target_spec
        row[f'Sens @ Spec={target_spec:.2f}'] = tpr[mask].max() if mask.any() else np.nan
    sens_rows.append(row)

sens_df = pd.DataFrame(sens_rows)
print(sens_df.round(4).to_string(index=False))
sens_df.to_csv(f"{FIG_DIR}/table_sensitivity_at_fixed_specificity.csv", index=False)
print(f"\n[Saved] table_sensitivity_at_fixed_specificity.csv")

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


# ============================================================
# Grad-CAM Extractor
# ============================================================

class GradCAMExtractor:
    """
    Manual Grad-CAM for ACLCrossAttentionNet's shared multi-view
    backbone (thesis 4.6).
    """

    def __init__(self, model):
        self.model = model

    def generate(
        self,
        sag,
        cor,
        ax,
        plane='sagittal'
    ):

        self.model.eval()

        # ----------------------------------------------------
        # Keep batch dimension and get spatial dimensions
        # ----------------------------------------------------
        b, s, h, w = sag.shape

        inputs = {
            'sagittal': sag,
            'coronal': cor,
            'axial': ax
        }

        if plane not in inputs:
            raise ValueError(
                f"Unknown plane: {plane}. "
                f"Expected one of {list(inputs.keys())}"
            )

        # ----------------------------------------------------
        # Move all inputs to device
        # ----------------------------------------------------
        sag_ = sag.to(DEVICE)
        cor_ = cor.to(DEVICE)
        ax_  = ax.to(DEVICE)

        # ----------------------------------------------------
        # IMPORTANT:
        # Only the selected plane requires gradient.
        # ----------------------------------------------------
        target_input = (
            inputs[plane]
            .clone()
            .detach()
            .to(DEVICE)
            .requires_grad_(True)
        )

        # Replace selected plane with target_input
        if plane == 'sagittal':
            sag_ = target_input
        elif plane == 'coronal':
            cor_ = target_input
        elif plane == 'axial':
            ax_ = target_input

        # ----------------------------------------------------
        # Backbone
        # ----------------------------------------------------
        fs = self.model.backbone(sag_)
        fc = self.model.backbone(cor_)
        fa = self.model.backbone(ax_)

        # ----------------------------------------------------
        # Select feature map for Grad-CAM
        # ----------------------------------------------------
        if plane == 'sagittal':
            feat = fs
        elif plane == 'coronal':
            feat = fc
        else:
            feat = fa

        feat.retain_grad()

        # ----------------------------------------------------
        # Tokenization
        # ----------------------------------------------------
        ts = self.model.tokenizer(fs, b, s)
        tc = self.model.tokenizer(fc, b, s)
        ta = self.model.tokenizer(fa, b, s)

        # ----------------------------------------------------
        # Slice attention
        # ----------------------------------------------------
        ts = self.model.slice_attn(ts)
        tc = self.model.slice_attn(tc)
        ta = self.model.slice_attn(ta)

        # ----------------------------------------------------
        # Cross attention blocks
        # ----------------------------------------------------
        ts, tc, ta = self.model.cross_block_1(
            ts, tc, ta
        )

        ts, tc, ta = self.model.cross_block_2(
            ts, tc, ta
        )

        # ----------------------------------------------------
        # Gated fusion
        # ----------------------------------------------------
        fused, gate_w = self.model.gated_fusion(
            ts, tc, ta
        )

        # ----------------------------------------------------
        # Context encoder
        # ----------------------------------------------------
        ctx = self.model.context_enc(fused)

        # Mean pooling
        vec = ctx.mean(dim=1)

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------
        logit = self.model.classifier(vec)

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------
        self.model.zero_grad(set_to_none=True)

        logit.squeeze().backward()

        # ----------------------------------------------------
        # Grad-CAM gradients
        # ----------------------------------------------------
        grads = feat.grad

        if grads is None:
            raise RuntimeError(
                "feat.grad is None. "
                "Grad-CAM gradient was not captured."
            )

        acts = feat.detach()

        # ----------------------------------------------------
        # Channel weights
        # ----------------------------------------------------
        weights = grads.mean(
            dim=(2, 3),
            keepdim=True
        )

        # ----------------------------------------------------
        # Grad-CAM
        # ----------------------------------------------------
        raw_cam = F.relu(
            (weights * acts).sum(dim=1)
        )

        # Expected:
        # [B*S, 7, 7]
        raw_cam = raw_cam.view(
            b * s,
            7,
            7
        )

        # ----------------------------------------------------
        # Upsample to original image size
        # ----------------------------------------------------
        cam_up = F.interpolate(
            raw_cam.unsqueeze(1),
            size=(h, w),
            mode='bilinear',
            align_corners=False
        ).squeeze(1)

        # ----------------------------------------------------
        # Normalize each slice independently
        # ----------------------------------------------------
        cam_up = cam_up.detach().cpu().numpy()

        for i in range(cam_up.shape[0]):

            c = cam_up[i]

            c_min = float(c.min())
            c_max = float(c.max())

            rng = c_max - c_min

            if rng > 0:
                cam_up[i] = (
                    (c - c_min) / rng
                )
            else:
                cam_up[i] = np.zeros_like(c)

        # ----------------------------------------------------
        # Probability
        # ----------------------------------------------------
        probability = torch.sigmoid(
            logit
        ).detach().cpu().item()

        # ----------------------------------------------------
        # Gate weights
        # ----------------------------------------------------
        gate_weights = (
            gate_w
            .detach()
            .cpu()
            .numpy()[0]
        )

        return (
            cam_up,
            probability,
            gate_weights
        )


# ============================================================
# Slice Attention Weights
# ============================================================

def get_slice_attention_weights(
    model,
    sag,
    cor,
    ax
):
    """
    Reads the slice-attention MLP's raw scores via a
    read-only forward hook.

    This is a non-gradient interpretability signal.
    """

    captured = {}

    def hook(module, inp, out):
        captured['logits'] = out.detach()

    h = model.slice_attn.mlp.register_forward_hook(hook)

    model.eval()

    try:

        with torch.no_grad():

            b, s, hh, ww = sag.shape

            weights_per_plane = {}

            for name, view in [
                ('sagittal', sag),
                ('coronal', cor),
                ('axial', ax)
            ]:

                # Backbone
                f = model.backbone(
                    view.to(DEVICE)
                )

                # Tokenizer
                t_raw = model.tokenizer(
                    f,
                    b,
                    s
                )

                # Slice attention
                _ = model.slice_attn(t_raw)

                # Get raw logits
                logits = captured['logits']

                # Softmax
                w = F.softmax(
                    logits,
                    dim=1
                )

                # Match original implementation
                w = (
                    w
                    .squeeze(-1)
                    .squeeze(0)
                    .mean(dim=-1)
                )

                weights_per_plane[name] = (
                    w.cpu().numpy()
                )

    finally:

        h.remove()

    return (
        weights_per_plane['sagittal'],
        weights_per_plane['coronal'],
        weights_per_plane['axial']
    )


# ============================================================
# Initialize Grad-CAM
# ============================================================

gradcam = GradCAMExtractor(
    proposed_model
)

print(
    "[Grad-CAM] Extractor ready "
    "(per-plane isolated, runs on this run's proposed_model)."
)


# ============================================================
# Case accessor
# ============================================================

def get_case(idx):

    sag, cor, ax, lbl = valid_ds[idx]

    return (
        sag.unsqueeze(0),
        cor.unsqueeze(0),
        ax.unsqueeze(0),
        lbl.item()
    )


# ============================================================
# IMPORTANT:
# CASE_IDX is DATASET INDEX
# ============================================================

CASE_IDX = 66


# ============================================================
# Check that index 66 actually exists
# ============================================================

if CASE_IDX < 0 or CASE_IDX >= len(valid_ds):

    raise IndexError(
        f"CASE_IDX={CASE_IDX} is outside valid_ds. "
        f"Dataset length = {len(valid_ds)}"
    )


# ============================================================
# Load Case 66
# ============================================================

sag_t, cor_t, ax_t, true_lbl = get_case(
    CASE_IDX
)


# ============================================================
# Print diagnostic information
# ============================================================

print()
print("=" * 70)
print("CASE SELECTION CHECK")
print("=" * 70)

print(f"Requested dataset index : {CASE_IDX}")
print(f"True label              : {true_lbl}")

print(
    f"Sagittal tensor shape   : "
    f"{tuple(sag_t.shape)}"
)

print(
    f"Coronal tensor shape    : "
    f"{tuple(cor_t.shape)}"
)

print(
    f"Axial tensor shape      : "
    f"{tuple(ax_t.shape)}"
)

print(
    f"Sagittal value range    : "
    f"{sag_t.min().item():.4f} -> "
    f"{sag_t.max().item():.4f}"
)

print(
    f"Coronal value range     : "
    f"{cor_t.min().item():.4f} -> "
    f"{cor_t.max().item():.4f}"
)

print(
    f"Axial value range       : "
    f"{ax_t.min().item():.4f} -> "
    f"{ax_t.max().item():.4f}"
)

print("=" * 70)
print()


# ============================================================
# Determine middle slice separately for each plane
# ============================================================

mid_sag = sag_t.shape[1] // 2
mid_cor = cor_t.shape[1] // 2
mid_ax  = ax_t.shape[1] // 2


print(
    f"[Case {CASE_IDX}] "
    f"Middle slices -> "
    f"Sagittal: {mid_sag}, "
    f"Coronal: {mid_cor}, "
    f"Axial: {mid_ax}"
)


# ============================================================
# Figure 4-4
# Single case, all 3 planes
# ============================================================

fig, axs = plt.subplots(
    2,
    3,
    figsize=(13, 8)
)


plane_inputs = {
    'sagittal': sag_t,
    'coronal': cor_t,
    'axial': ax_t
}


plane_mid = {
    'sagittal': mid_sag,
    'coronal': mid_cor,
    'axial': mid_ax
}


prob_demo = None


for col, plane in enumerate(
    ['sagittal', 'coronal', 'axial']
):

    print(
        f"[Grad-CAM] Processing "
        f"Case {CASE_IDX} - {plane}"
    )

    # --------------------------------------------------------
    # Generate Grad-CAM
    # --------------------------------------------------------

    cam, prob_demo, gw = gradcam.generate(
        sag_t,
        cor_t,
        ax_t,
        plane=plane
    )

    # --------------------------------------------------------
    # Correct slice for this plane
    # --------------------------------------------------------

    mid = plane_mid[plane]

    raw_img = plane_inputs[
        plane
    ][0, mid].detach().cpu().numpy()

    # --------------------------------------------------------
    # Normalize image
    # --------------------------------------------------------

    raw_min = raw_img.min()
    raw_max = raw_img.max()

    raw_img = (
        raw_img - raw_min
    ) / (
        raw_max - raw_min + 1e-8
    )

    # --------------------------------------------------------
    # Original image
    # --------------------------------------------------------

    axs[0, col].imshow(
        raw_img,
        cmap='gray'
    )

    axs[0, col].set_title(
        f"{plane.capitalize()}\n"
        f"Case {CASE_IDX} | Slice {mid}"
    )

    axs[0, col].axis('off')

    # --------------------------------------------------------
    # Grad-CAM overlay
    # --------------------------------------------------------

    axs[1, col].imshow(
        raw_img,
        cmap='gray'
    )

    axs[1, col].imshow(
        cam[mid],
        cmap='jet',
        alpha=0.45
    )

    axs[1, col].set_title(
        f"Grad-CAM | p={prob_demo:.2f}\n"
        f"gate={gw[col]:.2f}"
    )

    axs[1, col].axis('off')


# ============================================================
# Overall title
# ============================================================

fig.suptitle(
    f"Grad-CAM — Case {CASE_IDX} "
    f"(True label = {true_lbl})",
    fontsize=15,
    y=1.02
)


plt.tight_layout()


# ============================================================
# Save Figure 4-4
# ============================================================

os.makedirs(
    FIG_DIR,
    exist_ok=True
)

fig_path_44 = os.path.join(
    FIG_DIR,
    "fig_4-4_gradcam_3plane_case66.png"
)


plt.savefig(
    fig_path_44,
    dpi=200,
    bbox_inches='tight'
)

plt.show()


print(
    f"[Saved] {fig_path_44}"
)

print(
    f"[Figure 4-4] "
    f"Displayed dataset index = {CASE_IDX}"
)


# ============================================================
# Figure 4-5
# Multiple correctly detected cases
# Sagittal plane
# ============================================================

probs_arr = np.array(
    proposed_probs
)

labels_arr = np.array(
    proposed_labels
)


# ------------------------------------------------------------
# True Positives
# ------------------------------------------------------------

tp_mask = (
    (labels_arr == 1) &
    (probs_arr > best_thr)
)


# ------------------------------------------------------------
# False Negatives
# ------------------------------------------------------------

fn_mask = (
    (labels_arr == 1) &
    (probs_arr <= best_thr)
)


# ------------------------------------------------------------
# True Negatives
# ------------------------------------------------------------

tn_mask = (
    (labels_arr == 0) &
    (probs_arr <= best_thr)
)


# ------------------------------------------------------------
# Sort indices
# ------------------------------------------------------------

tp_indices = np.where(
    tp_mask
)[0]

fn_indices = np.where(
    fn_mask
)[0]

tn_indices = np.where(
    tn_mask
)[0]


tp_idx_sorted = (
    tp_indices[
        np.argsort(
            -probs_arr[tp_indices]
        )
    ]
)


fn_idx_sorted = fn_indices


tn_idx_sorted = (
    tn_indices[
        np.argsort(
            probs_arr[tn_indices]
        )
    ]
)


print()
print(
    f"[Grad-CAM] True Positives  : "
    f"{len(tp_idx_sorted)}"
)

print(
    f"[Grad-CAM] False Negatives : "
    f"{len(fn_idx_sorted)}"
)

print(
    f"[Grad-CAM] True Negatives  : "
    f"{len(tn_idx_sorted)}"
)


# ============================================================
# Number of examples
# ============================================================

n_show = min(
    6,
    len(tp_idx_sorted)
)


# ============================================================
# Create Figure 4-5
# ============================================================

fig, axs = plt.subplots(
    2,
    3,
    figsize=(13, 8)
)


for k in range(n_show):

    idx = int(
        tp_idx_sorted[k]
    )

    sag_k, cor_k, ax_k, label_k = get_case(
        idx
    )

    cam_k, prob_k, gw_k = gradcam.generate(
        sag_k,
        cor_k,
        ax_k,
        plane='sagittal'
    )

    mid_k = (
        sag_k.shape[1] // 2
    )

    raw_k = (
        sag_k[0, mid_k]
        .detach()
        .cpu()
        .numpy()
    )

    raw_k = (
        raw_k - raw_k.min()
    ) / (
        raw_k.max() -
        raw_k.min() +
        1e-8
    )

    r, c = divmod(
        k,
        3
    )

    axs[r, c].imshow(
        raw_k,
        cmap='gray'
    )

    axs[r, c].imshow(
        cam_k[mid_k],
        cmap='jet',
        alpha=0.45
    )

    axs[r, c].set_title(
        f"Case #{idx} | "
        f"p={prob_k:.2f}",
        fontsize=10
    )

    axs[r, c].axis('off')


# ============================================================
# Hide unused axes
# ============================================================

for k in range(
    n_show,
    6
):

    r, c = divmod(
        k,
        3
    )

    axs[r, c].axis('off')


# ============================================================
# Figure title
# ============================================================

plt.suptitle(
    "Sagittal Grad-CAM Across "
    "Multiple Correctly-Detected ACL Tears",
    y=1.02,
    fontsize=13
)

plt.tight_layout()


# ============================================================
# Save Figure 4-5
# ============================================================

fig_path_45 = os.path.join(
    FIG_DIR,
    "fig_4-5_gradcam_multicase_grid.png"
)


plt.savefig(
    fig_path_45,
    dpi=200,
    bbox_inches='tight'
)

plt.show()


print(
    f"[Saved] {fig_path_45}"
)

print(
    "[Saved] "
    "fig_4-5_gradcam_multicase_grid.png "
    "-> Thesis Figure 4-5"
)